In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

# Load the custom-trained YOLO model
model = YOLO("F:/insighteye/datacenter/Realtime-Accident-Detection-Yolov8-main/best.pt")  # Replace with your custom model path

# Function to calculate the centroid of a bounding box
def get_centroid(box):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2
    return (cx, cy)

# Function to calculate the distance between two centroids
def distance(centroid1, centroid2):
    return np.sqrt((centroid1[0] - centroid2[0]) ** 2 + (centroid1[1] - centroid2[1]) ** 2)

# Function to combine two bounding boxes into one larger bounding box
def combine_boxes(box1, box2):
    x1 = min(box1[0], box2[0])
    y1 = min(box1[1], box2[1])
    x2 = max(box1[2], box2[2])
    y2 = max(box1[3], box2[3])
    return (x1, y1, x2, y2)

# Initialize video capture
video_path = "F:/insighteye/datacenter/accident detection/accident2.mp4"  # Path to your CCTV video file
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Video processing complete.")
        break

    # Run YOLO detection on the current frame
    results = model(frame)

    persons = []
    vehicles = []
    centroids_persons = []
    centroids_vehicles = []

    # Extract detections
    for result in results:
        for box, conf, cls in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
            x1, y1, x2, y2 = map(int, box[:4])
            conf = conf.item()
            cls = int(cls.item())

            # Adjust the class IDs based on your custom model
            if conf > 0.5:  # Confidence threshold
                if cls == 0:  # Assuming class 0 is Person
                    persons.append((x1, y1, x2, y2))
                    centroids_persons.append(get_centroid((x1, y1, x2, y2)))
                elif cls == 1:  # Assuming class 1 is Vehicle
                    vehicles.append((x1, y1, x2, y2))
                    centroids_vehicles.append(get_centroid((x1, y1, x2, y2)))

    accident_detected = False
    combined_box = None

    # Check for collisions based on centroid distances
    for i, person_centroid in enumerate(centroids_persons):
        for j, vehicle_centroid in enumerate(centroids_vehicles):
            if distance(person_centroid, vehicle_centroid) < 50:  # Adjust the threshold as needed
                accident_detected = True
                combined_box = combine_boxes(persons[i], vehicles[j])
                break
        if accident_detected:
            break

    # Draw bounding boxes
    for box in persons:
        cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 1)  # Green for persons

    for box in vehicles:
        cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (255, 0, 0), 1)  # Blue for vehicles

    if accident_detected and combined_box:
        # Draw combined red bounding box for accident
        cv2.rectangle(frame, (combined_box[0], combined_box[1]), (combined_box[2], combined_box[3]), (0, 0, 255), 2)
        cv2.putText(frame, "Accident Detected", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Display the processed video frame
    cv2.imshow("Accident Detection", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()



0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 309.7ms
Speed: 2.0ms preprocess, 309.7ms inference, 1.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 407.0ms
Speed: 2.0ms preprocess, 407.0ms inference, 1.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 292.3ms
Speed: 2.0ms preprocess, 292.3ms inference, 1.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 282.3ms
Speed: 2.0ms preprocess, 282.3ms inference, 2.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 262.3ms
Speed: 2.0ms preprocess, 262.3ms inference, 2.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 272.8ms
Speed: 3.0ms preprocess, 272.8ms inference, 2.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 bikes, 2 cars, 1 car_car_accident, 264.0ms
Speed: 1.0ms 